In [25]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt


def process_image_with_contours_and_hough(image_path, output_dir=None):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"❌ Error: Could not read image at {image_path}")
        return

    # === 二值化與形態學處理 ===
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.dilate(binary, kernel, iterations=3)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=5)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)

    # === 找輪廓並去除雜訊 ===
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_area = 1000
    filtered = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]

    if not filtered:
        print(f"⚠️ No valid contours found in {image_path}")
        return

    main_contour = max(filtered, key=cv2.contourArea)

    # === 平滑輪廓 ===
    epsilon = 0.002 * cv2.arcLength(main_contour, True)  # 可調整為 0.005 ~ 0.02
    main_contour = cv2.approxPolyDP(main_contour, epsilon, True)

    # === 建立輪廓遮罩 ===
    mask = np.zeros_like(binary)
    cv2.drawContours(mask, [main_contour], -1, 255, thickness=cv2.FILLED)

    # === 霍夫轉換 ===
    masked = cv2.bitwise_and(binary, mask)
    edges = cv2.Canny(binary, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180, threshold=10, minLineLength=10, maxLineGap=100
    )

    # === 繪圖 ===
    img_color = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(img_color, [main_contour], -1, (0, 255, 0), 2)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(img_color, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # === 輸出或顯示 ===
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, os.path.basename(image_path))
        cv2.imwrite(output_path, img_color)
    else:
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
        plt.title(os.path.basename(image_path))
        plt.axis("off")
        plt.show()


def batch_process_folder(folder_path, output_dir=None):
    supported_ext = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
    for filename in os.listdir(folder_path):
        if any(filename.lower().endswith(ext) for ext in supported_ext):
            image_path = os.path.join(folder_path, filename)
            print(f"📌 處理中：{filename}")
            process_image_with_contours_and_hough(image_path, output_dir)


# === 執行區 ===
folder_path = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\202508\RB"
output_path = os.path.join(folder_path, "output_hough_lines")

batch_process_folder(folder_path, output_path)

📌 處理中：0.3_0.3_30_01_RB(1024).png
📌 處理中：0.3_0.3_30_02_RB(1024).png
📌 處理中：0.3_0.3_60_01_RB(1024).png
📌 處理中：0.3_0.3_60_02_RB(1024).png
📌 處理中：0.3_0.3_90_01_RB(1024).png
📌 處理中：0.3_0.3_90_02_RB(1024).png
📌 處理中：0.3_0.6_30_01_RB(1024).png
📌 處理中：0.3_0.6_60_01_RB(1024).png
📌 處理中：0.3_0.6_60_02_RB(1024).png
📌 處理中：0.3_0.9_30_01-1_RB(1024).png
📌 處理中：0.3_0.9_30_01_RB(1024).png
📌 處理中：0.3_0.9_30_02_RB(1024).png
📌 處理中：0.3_0.9_60_01_RB(1024).png
📌 處理中：0.3_0.9_90_01_RB(1024).png
📌 處理中：0.3_06_90_01(1024).png
📌 處理中：0.6_0.3_30_01_RB(1024).png
📌 處理中：0.6_0.3_60_01_RB(1024).png
📌 處理中：0.6_0.3_60_02_RB(1024).png
📌 處理中：0.6_0.3_90_01_RB(1024).png
📌 處理中：0.6_0.3_90_02_RB(1024).png
📌 處理中：0.9_0.3_60_01_RB(2048).png
📌 處理中：0.9_0.3_60_02_RB(2048).png
📌 處理中：0.9_0.3_60_03_RB(2048).png
📌 處理中：0.9_0.3_60_04_RB(2048).png
📌 處理中：0.9_0.3_60_05_RB(2048).png
📌 處理中：0.9_0.3_60_06_RB(1024).png
📌 處理中：0.9_0.9_120.png
